In [1]:
import pandas as pd
import os
import shutil
import numpy as np
import glob
import sys

In [ ]:
### EUR

In [ ]:
pheno = pd.read_csv(f'{DATA_DIR}/imputed_genotypes/EUR/chr1_EUR_release9_vwb.psam', sep='\t') 
pheno.head()

In [ ]:
pcs = pd.read_csv(f'{DATA_DIR}/raw_genotypes/EUR/EUR_release9_vwb.eigenvec', sep='\t')
pcs.head()

In [ ]:
merge1 = pd.merge(pheno, pcs, left_on='#IID', right_on='IID')
merge1.head()

In [ ]:
# remove diagnosis changes

In [ ]:
change = pd.read_csv(f'{DATA_DIR}/clinical_data/r9_extended_clinical_data_vwb.csv')
change = change[['GP2ID','visit_month', 'age_at_baseline', 'primary_diagnosis', 'last_diagnosis']]

In [ ]:
no_dup = change.dropna(subset=['last_diagnosis'])
no_dup['diagnosis_change'] = no_dup.apply(lambda row: 'No' if row['primary_diagnosis'] == row['last_diagnosis'] else 'Yes', axis=1)
d_change = no_dup[no_dup['diagnosis_change']=='Yes']
eur_d = pd.merge(d_change, merge1, left_on='GP2ID', right_on='#IID')
diag_remove = eur_d['GP2ID']
eur_updated = merge1[~merge1['IID'].isin(diag_remove)]

In [ ]:
#remove related people

In [ ]:
related = pd.read_csv(f'{DATA_DIR}/meta_data/related_samples/EUR_release9_vwb.related')
remove = related['IID1']
merge_norel = eur_updated[~eur_updated['IID'].isin(remove)]
merge_norel.info()

In [ ]:
# add popualtion controls in

In [ ]:
master = pd.read_csv(f'{DATA_DIR}/clinical_data/master_key_release9_final_vwb.csv')

In [ ]:
pop_control = master[master['baseline_GP2_phenotype']=='Population Control']
pop_change = pop_control['GP2ID']
merge_norel['PHENO1'][merge_norel['#IID'].isin(pop_change)] = 1
merge_norel.info()

In [ ]:
merge_pheno = merge_norel[~merge_norel['PHENO1'].isnull()]

In [ ]:
# add age

In [ ]:
master['age_of_onset'].fillna(master['age_at_sample_collection'], inplace=True)
master['age_of_onset'].fillna(master['age_at_diagnosis'], inplace=True)
test = pd.merge(master, merge_pheno, left_on='GP2ID', right_on='IID')

In [ ]:
merge1_final = test[['IID', 'PHENO1', 'SEX', 'age_of_onset', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']]
merge1_final.columns = ['IID', 'PHENO1', 'SEX', 'AGE', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']
merge1_final.fillna('-9', inplace=True)

In [ ]:
merge1_final.to_csv('GP2_EUR_covars.txt', index=False, sep='\t')

In [ ]:
# Create shell script to execute
count = 0 # for Job ID count

# Establish paths for inputs/outputs


release_path = f'{DATA_DIR}/imputed_genotypes/EUR/chr${{CHROM}}_EUR_release9_vwb'
out_dir = f'chr${{CHROM}}.R9_MALE_GWAS_RESULTS' 

subset_cmd = f"/home/leonardhl/bin/plink2/plink2 --pfile {release_path} \
     --glm hide-covar no-x-sex firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+err \
     --pheno-name PHENO1 --covar-variance-standardize \
     --keep-if SEX==1 \
     --pheno GP2_EUR_covars.txt \
     --covar GP2_EUR_covars.txt \
     --covar-name SEX,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \
     --out {out_dir}"

with open(f'gwas_plink_male_r9.sh', 'w') as f:
    f.write('#!/bin/bash\n\n')
    f.write('CHROM=$1\n\n')
    f.write(f'{subset_cmd}\n')
    f.close()



In [ ]:
# Create shell script to execute
count = 0 # for Job ID count

# Establish paths for inputs/outputs


release_path = f'{DATA_DIR}/imputed_genotypes/EUR/chr${{CHROM}}_EUR_release9_vwb'
out_dir = f'chr${{CHROM}}.R9_FEMALE_GWAS_RESULTS' 

subset_cmd = f"{TOOLS_DIR}/plink2 --pfile {release_path} \
     --glm hide-covar no-x-sex firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+err \
     --pheno-name PHENO1 --covar-variance-standardize \
     --keep-if SEX==2 \
     --pheno GP2_EUR_covars.txt \
     --covar GP2_EUR_covars.txt \
     --covar-name SEX,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \
     --out {out_dir}"

with open(f'gwas_plink_female_r9.sh', 'w') as f:
    f.write('#!/bin/bash\n\n')
    f.write('CHROM=$1\n\n')
    f.write(f'{subset_cmd}\n')
    f.close()



In [ ]:
### AJ

In [ ]:
pheno = pd.read_csv(f'{DATA_DIR}/imputed_genotypes/AJ/chr1_AJ_release9_vwb.psam', sep='\t') 
pheno.head()

In [ ]:
pcs = pd.read_csv(f'{DATA_DIR}/raw_genotypes/AJ/AJ_release9_vwb.eigenvec', sep='\t')
pcs.head()

In [ ]:
merge1 = pd.merge(pheno, pcs, left_on='#IID', right_on='IID')
merge1.head()

In [ ]:
change = pd.read_csv(f'{DATA_DIR}/clinical_data/r9_extended_clinical_data_vwb.csv')
change = change[['GP2ID','visit_month', 'age_at_baseline', 'primary_diagnosis', 'last_diagnosis']]

In [ ]:
no_dup = change.dropna(subset=['last_diagnosis'])
no_dup['diagnosis_change'] = no_dup.apply(lambda row: 'No' if row['primary_diagnosis'] == row['last_diagnosis'] else 'Yes', axis=1)
d_change = no_dup[no_dup['diagnosis_change']=='Yes']
aj_d = pd.merge(d_change, merge1, left_on='GP2ID', right_on='#IID')
diag_remove = aj_d['GP2ID']
aj_updated = merge1[~merge1['IID'].isin(diag_remove)]

In [ ]:
related = pd.read_csv(f'{DATA_DIR}/meta_data/related_samples/AJ_release9_vwb.related')
remove = related['IID1']
merge_norel = aj_updated[~aj_updated['IID'].isin(remove)]
merge_norel.info()

In [ ]:
master = pd.read_csv(f'{DATA_DIR}/clinical_data/master_key_release9_final_vwb.csv')

In [ ]:
pop_control = master[master['baseline_GP2_phenotype']=='Population Control']
pop_change = pop_control['GP2ID']
merge_norel['PHENO1'][merge_norel['#IID'].isin(pop_change)] = 1
merge_norel.info()

In [ ]:
merge_pheno = merge_norel[~merge_norel['PHENO1'].isnull()]

In [ ]:
master['age_of_onset'].fillna(master['age_at_sample_collection'], inplace=True)
master['age_of_onset'].fillna(master['age_at_diagnosis'], inplace=True)
test = pd.merge(master, merge_pheno, left_on='GP2ID', right_on='IID')

In [ ]:
merge1_final = test[['IID', 'PHENO1', 'SEX', 'age_of_onset', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']]
merge1_final.columns = ['IID', 'PHENO1', 'SEX', 'AGE', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']
merge1_final.fillna('-9', inplace=True)

In [ ]:
merge1_final.to_csv('GP2_AJ_covars.txt', index=False, sep='\t')

In [ ]:
# Create shell script to execute
count = 0 # for Job ID count

# Establish paths for inputs/outputs


release_path = f'{DATA_DIR}/imputed_genotypes/AJ/chr${{CHROM}}_AJ_release9_vwb'
out_dir = f'chr${{CHROM}}.AJ_R9_MALE_GWAS_RESULTS' 

subset_cmd = f"{TOOLS_DIR}/plink2 --pfile {release_path} \
     --glm hide-covar no-x-sex firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+err \
     --pheno-name PHENO1 --covar-variance-standardize \
     --keep-if SEX==1 \
     --pheno GP2_AJ_covars.txt \
     --covar GP2_AJ_covars.txt \
     --covar-name SEX,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \
     --out {out_dir}"

with open(f'aj_gwas_plink_male_r9.sh', 'w') as f:
    f.write('#!/bin/bash\n\n')
    f.write('CHROM=$1\n\n')
    f.write(f'{subset_cmd}\n')
    f.close()



In [ ]:
# Create shell script to execute
count = 0 # for Job ID count

# Establish paths for inputs/outputs


release_path = f'{DATA_DIR}/imputed_genotypes/AJ/chr${{CHROM}}_AJ_release9_vwb'
out_dir = f'chr${{CHROM}}.AJ_R9_FEMALE_GWAS_RESULTS' 

subset_cmd = f"{TOOLS_DIR}/plink2 --pfile {release_path} \
     --glm hide-covar no-x-sex firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+err \
     --pheno-name PHENO1 --covar-variance-standardize \
     --keep-if SEX==2 \
     --pheno GP2_AJ_covars.txt \
     --covar GP2_AJ_covars.txt \
     --covar-name SEX,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \
     --out {out_dir}"

with open(f'aj_gwas_plink_female_r9.sh', 'w') as f:
    f.write('#!/bin/bash\n\n')
    f.write('CHROM=$1\n\n')
    f.write(f'{subset_cmd}\n')
    f.close()



In [ ]:
### R10 replication

In [ ]:
pheno = pd.read_csv(f'{DATA_DIR}/imputed_genotypes/EUR/chr1_EUR_release10_vwb.psam', sep='\t') 
pheno.head()

In [ ]:
pcs = pd.read_csv(f'{DATA_DIR}/raw_genotypes/EUR/EUR_release10_vwb.eigenvec', sep='\t')
pcs.head()

In [ ]:
merge1 = pd.merge(pheno, pcs, left_on='#IID', right_on='IID')
merge1.head()

In [ ]:
# remove diagnosis changes

In [ ]:
change = pd.read_csv(f'{DATA_DIR}/clinical_data/r10_extended_clinical_data_vwb.csv')
change = change[['GP2ID','visit_month', 'age_at_baseline', 'primary_diagnosis', 'last_diagnosis']]

In [ ]:
no_dup = change.dropna(subset=['last_diagnosis'])
no_dup['diagnosis_change'] = no_dup.apply(lambda row: 'No' if row['primary_diagnosis'] == row['last_diagnosis'] else 'Yes', axis=1)
d_change = no_dup[no_dup['diagnosis_change']=='Yes']
eur_d = pd.merge(d_change, merge1, left_on='GP2ID', right_on='#IID')
diag_remove = eur_d['GP2ID']
eur_updated = merge1[~merge1['IID'].isin(diag_remove)]

In [ ]:
#remove related people

In [ ]:
related = pd.read_csv(f'{DATA_DIR}/meta_data/related_samples/EUR_release10_vwb.related')
remove = related['IID1']
merge_norel = eur_updated[~eur_updated['IID'].isin(remove)]
merge_norel.info()

In [ ]:
# add popualtion controls in

In [ ]:
master = pd.read_csv(f'{DATA_DIR}/clinical_data/master_key_release10_final_vwb.csv')

In [ ]:
pop_control = master[master['baseline_GP2_phenotype']=='Population Control']
pop_change = pop_control['GP2ID']
merge_norel['PHENO1'][merge_norel['#IID'].isin(pop_change)] = 1
merge_norel.info()

In [ ]:
merge_pheno = merge_norel[~merge_norel['PHENO1'].isnull()]

In [ ]:
# add age

In [ ]:
master['age_of_onset'].fillna(master['age_at_sample_collection'], inplace=True)
master['age_of_onset'].fillna(master['age_at_diagnosis'], inplace=True)
test = pd.merge(master, merge_pheno, left_on='GP2ID', right_on='IID')

In [ ]:
merge1_final = test[['IID', 'PHENO1', 'SEX', 'age_of_onset', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']]
merge1_final.columns = ['IID', 'PHENO1', 'SEX', 'AGE', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']
merge1_final.fillna('-9', inplace=True)

In [ ]:
# only R10 samples

In [ ]:
r9_samples = pd.read_csv(f'{DATA_DIR}/meta_data/previous_release_samples/release9_EUR_vwb.samples', sep='\t', header=None)
r9_samples.columns = ['GP2ID']

In [ ]:
merge1_final_r10 = merge1_final[~merge1_final['IID'].isin(r9_samples['GP2ID'])]
merge1_final_r10.to_csv('R10_ONLY_GP2_EUR_covars.txt', index=False, sep='\t')

In [ ]:
# Create shell script to execute
count = 0 # for Job ID count

# Establish paths for inputs/outputs


release_path = f'{DATA_DIR}/imputed_genotypes/EUR/chr${{CHROM}}_EUR_release10_vwb'
out_dir = f'chr${{CHROM}}.R10_REP_MALE_GWAS_RESULTS' 

subset_cmd = f"{TOOLS_DIR}/plink2 --pfile {release_path} \
     --glm hide-covar no-x-sex firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+err \
     --pheno-name PHENO1 --covar-variance-standardize \
     --keep-if SEX==1 \
     --pheno R10_ONLY_GP2_EUR_covars.txt \
     --covar R10_ONLY_GP2_EUR_covars.txt \
     --covar-name SEX,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \
     --out {out_dir}"

with open(f'gwas_plink_male_r10_rep.sh', 'w') as f:
    f.write('#!/bin/bash\n\n')
    f.write('CHROM=$1\n\n')
    f.write(f'{subset_cmd}\n')
    f.close()



In [ ]:
# Create shell script to execute
count = 0 # for Job ID count

# Establish paths for inputs/outputs


release_path = f'{DATA_DIR}/imputed_genotypes/EUR/chr${{CHROM}}_EUR_release10_vwb'
snp = f'chr${{SNP}}'
out_dir = f'chr${{CHROM}}.R10_REP_FEMALE_GWAS_RESULTS' 

subset_cmd = f"{TOOLS_DIR}/plink2 --pfile {release_path} \
     --glm hide-covar no-x-sex firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+err \
     --pheno-name PHENO1 --covar-variance-standardize \
     --keep-if SEX==2 \
     --pheno R10_ONLY_GP2_EUR_covars.txt \
     --covar R10_ONLY_GP2_EUR_covars.txt \
     --covar-name SEX,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \
     --out {out_dir}"

with open(f'gwas_plink_female_r10_rep.sh', 'w') as f:
    f.write('#!/bin/bash\n\n')
    f.write('CHROM=$1\n\n')
    f.write(f'{subset_cmd}\n')
    f.close()

In [ ]:
### control gwas

In [ ]:
# Create shell script to execute
count = 0 # for Job ID count

# Establish paths for inputs/outputs


release_path = f'{DATA_DIR}/imputed_genotypes/EUR/chr${{CHROM}}_EUR_release9_vwb'
out_dir = f'chr${{CHROM}}.CONTROL_SEX_GWAS_RESULTS' 

subset_cmd = f"{TOOLS_DIR}/plink2 --pfile {release_path} \
     --glm hide-covar no-x-sex firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+err \
     --pheno-name SEX --covar-variance-standardize \
     --keep-if PHENO1==1 \
     --pheno GP2_EUR_covars.txt \
     --covar GP2_EUR_covars.txt \
     --covar-name PHENO1,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \
     --out {out_dir}"

with open(f'gwas_plink_control.sh', 'w') as f:
    f.write('#!/bin/bash\n\n')
    f.write('CHROM=$1\n\n')
    f.write(f'{subset_cmd}\n')
    f.close()

